In [19]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
from datetime import datetime
from typing import Optional, Dict
import pandas as pd


class Alert(ABC):

    @property
    @abstractmethod
    def id(self) -> int: ...

    @abstractmethod
    def should_raise(self) -> bool: ...

    @abstractmethod
    def describe(self) -> str: ...


@dataclass
class Pathogen:
    org_id: Optional[int]
    org_name: str
    danger_weight: float
    time_window_days: int
    ward_thresholds: Dict[str, int] 
    staff_threshold: int

    def get_ward_threshold(self, ward_size: int) -> int:
        if ward_size <= 5:
            return self.ward_thresholds["5"]
        elif ward_size <= 10:
            return self.ward_thresholds["10"]
        else:
            return self.ward_thresholds["20"]


@dataclass
class MicrobiologyAlert(Alert):
    counter_id: int
    pathogen: Pathogen
    ward_id: int
    ward_size: int
    start_time: datetime
    alert_type: str
    curr_patient_number: int

    @property
    def id(self) -> int:
        return self.counter_id

    @property
    def org_id(self) -> Optional[int]:
        return self.pathogen.org_id

    def should_raise(self) -> bool:
        return self.curr_patient_number >= self.pathogen.get_ward_threshold(self.ward_size)

    def describe(self) -> str:
        thresh = self.pathogen.get_ward_threshold(self.ward_size)
        status = "⚠ ALERT" if self.should_raise() else "OK"
        return (
            f"[{status}][{self.alert_type}] Ward {self.ward_id} "
            f"({self.ward_size} beds) | {self.pathogen.org_name} | "
            f"{self.curr_patient_number}/{thresh} patients @ {self.start_time}"
        )


@dataclass
class WardAlert(Alert):
    counter_id: int
    ward_id: int
    pathogen: Pathogen
    ward_size: int
    start_time: datetime
    alert_type: str
    curr_patient_number: int

    @property
    def id(self) -> int:
        return self.counter_id

    def should_raise(self) -> bool:
        return self.curr_patient_number >= self.pathogen.get_ward_threshold(self.ward_size)

    def describe(self) -> str:
        thresh = self.pathogen.get_ward_threshold(self.ward_size)
        status = "⚠ ALERT" if self.should_raise() else "OK"
        return (
            f"[{status}][{self.alert_type}] Ward {self.ward_id} "
            f"({self.ward_size} beds) | {self.pathogen.org_name} | "
            f"{self.curr_patient_number}/{thresh} patients @ {self.start_time}"
        )


PATHOGEN_REGISTRY = {
    "CLOSTRIDIUM DIFFICILE": Pathogen(
        org_id=None, org_name="CLOSTRIDIUM DIFFICILE",
        danger_weight=3.0, time_window_days=3,
        ward_thresholds={"5": 1, "10": 1, "20": 1},
        staff_threshold=2,
    ),
    "GRAM NEGATIVE ROD": Pathogen(
        org_id=None, org_name="GRAM NEGATIVE ROD",
        danger_weight=1.5, time_window_days=2,
        ward_thresholds={"5": 1, "10": 1, "20": 2},
        staff_threshold=3,
    ),
    "CANDIDA ALBICANS": Pathogen(
        org_id=None, org_name="CANDIDA ALBICANS",
        danger_weight=1.0, time_window_days=2,
        ward_thresholds={"5": 1, "10": 1, "20": 2},
        staff_threshold=3,
    ),
    "STREPTOCOCCUS": Pathogen(
        org_id=None, org_name="STREPTOCOCCUS",
        danger_weight=1.0, time_window_days=2,
        ward_thresholds={"5": 1, "10": 2, "20": 3},
        staff_threshold=4,
    ),
    "HAEMOPHILUS INFLUENZAE": Pathogen(
        org_id=None, org_name="HAEMOPHILUS INFLUENZAE",
        danger_weight=1.0, time_window_days=2,
        ward_thresholds={"5": 1, "10": 1, "20": 2},
        staff_threshold=3,
    ),
    "YEAST unspeciated": Pathogen(
        org_id=None, org_name="YEAST unspeciated",
        danger_weight=1.0, time_window_days=3,
        ward_thresholds={"5": 1, "10": 1, "20": 3},
        staff_threshold=4,
    ),
    "ACINETOBACTER BAUMANNII COMPLEX": Pathogen(
        org_id=None, org_name="ACINETOBACTER BAUMANNII COMPLEX",
        danger_weight=3.0, time_window_days=3,
        ward_thresholds={"5": 1, "10": 1, "20": 1},
        staff_threshold=2
    ),
    "ASPERGILLUS SP. NOT FUMIGATUS, FLAVUS OR NIGER": Pathogen(
        org_id=None, org_name="ASPERGILLUS SP. NOT FUMIGATUS, FLAVUS OR NIGER", 
        danger_weight=1.5, time_window_days=2,
        ward_thresholds={"5": 1, "10": 1, "20": 2},
        staff_threshold=2
    ),
    "CORYNEBACTERIUM SPECIES (DIPHTHEROIDS)": Pathogen(
        org_id=None, org_name="CORYNEBACTERIUM SPECIES (DIPHTHEROIDS)",
        danger_weight=2, time_window_days=2,
        ward_thresholds={"5": 1, "10": 2, "20": 3},
        staff_threshold=2
    ),
    "ESCHERICHIA COLI": Pathogen(
        org_id=None, org_name="ESCHERICHIA COLI",
        danger_weight=3, time_window_days=3,
        ward_thresholds={"5": 1, "10": 1, "20": 2},
        staff_threshold=1
    ),
    "KLEBSIELLA PNEUMONIAE": Pathogen(
        org_id=None, org_name="KLEBSIELLA PNEUMONIAE", 
        danger_weight=3, time_window_days=1,
        ward_thresholds={"5": 1, "10": 1, "20": 1},
        staff_threshold=1
    ),
    "MYCELIA STERILIA": Pathogen(
        org_id=None, org_name="MYCELIA STERILIA", 
        danger_weight=1, time_window_days=4, 
        ward_thresholds={"5": 2, "10": 3, "20": 4},
        staff_threshold=4
    ),
    "MYCOBACTERIUM AVIUM COMPLEX": Pathogen(
        org_id=None, org_name="MYCOBACTERIUM AVIUM COMPLEX",
        danger_weight=1.5, time_window_days=1,
        ward_thresholds={"5": 1, "10": 2, "20": 4},
        staff_threshold=3
    ),
    "POSITIVE FOR METHICILLIN RESISTANT STAPH AUREUS": Pathogen(
        org_id=None, org_name="POSITIVE FOR METHICILLIN RESISTANT STAPH AUREUS",
        danger_weight=3, time_window_days=1,
        ward_thresholds={"5": 1, "10": 1, "20": 2},
        staff_threshold=1
    ),
    "PROTEUS MIRABILIS": Pathogen(
        org_id=None, org_name="PROTEUS MIRABILIS",
        danger_weight=1.5, time_window_days=1,
        ward_thresholds={"5": 2, "10": 3, "20": 4},
        staff_threshold=3
    ),
    "PSEUDOMONAS AERUGINOSA": Pathogen(
        org_id=None, org_name="PSEUDOMONAS AERUGINOSA",
        danger_weight=3, time_window_days=1,
        ward_thresholds={"5": 1, "10": 1, "20": 2},
        staff_threshold=1
    ),
    "STAPH AUREUS COAG +": Pathogen(
        org_id=None, org_name="STAPH AUREUS COAG +",
        danger_weight=3, time_window_days=1,
        ward_thresholds={"5": 1, "10": 1, "20": 2},
        staff_threshold=1
    ),
    "STAPHYLOCOCCUS, COAGULASE NEGATIVE": Pathogen(
        org_id=None, org_name="STAPHYLOCOCCUS, COAGULASE NEGATIVE",


danger_weight=1.5, time_window_days=1,
        ward_thresholds={"5": 2, "10": 2, "20": 3},
        staff_threshold=2
    ),
    "GRAM NEGATIVE ROD #2": Pathogen(
        org_id=None, org_name="GRAM NEGATIVE ROD #2",
        danger_weight=2, time_window_days=2,
        ward_thresholds={"5": 1, "10": 2, "20": 3},
        staff_threshold=2
    ),
    "GRAM NEGATIVE ROD #3": Pathogen(
        org_id=None, org_name="GRAM NEGATIVE ROD #3",
        danger_weight=2, time_window_days=2,
        ward_thresholds={"5": 1, "10": 2, "20": 3},
        staff_threshold=2
    ),
    "GRAM NEGATIVE ROD #4": Pathogen(
        org_id=None, org_name="GRAM NEGATIVE ROD #4",
        danger_weight=2, time_window_days=2,
        ward_thresholds={"5": 1, "10": 2, "20": 3},
        staff_threshold=2
    ),
}


In [20]:
import sqlite3
import pandas as pd

# Assuming db_path and desired_tables are defined earlier, e.g.:
db_path = '/Volumes/T7/OOP_project/OOP_database.db'  # From your MIMIC/OOP project [cite:16]
desired_tables = ['PATIENTS', 'CAREGIVERS', 'D_ITEMS', 'ADMISSIONS', 'ICUSTAYS', 'NOTEEVENTS', 'MICROBIOLOGYEVENTS', 'TRANSFERS', 'OUTPUTEVENTS', 'CHARTEVENTS']

from contextlib import contextmanager

@contextmanager
def get_db_connection(db_path):
    conn = sqlite3.connect(db_path)
    try:
        yield conn
    finally:
        conn.close()

with get_db_connection(db_path) as conn:
    db_tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table'", conn
    )["name"].str.upper().tolist()

for tbl in desired_tables:
    if tbl in db_tables:
        print(f"Table {tbl} exists.")  # Or your logic here
    else:
        print(f"Table {tbl} missing.")


Table PATIENTS exists.
Table CAREGIVERS exists.
Table D_ITEMS exists.
Table ADMISSIONS exists.
Table ICUSTAYS exists.
Table NOTEEVENTS exists.
Table MICROBIOLOGYEVENTS exists.
Table TRANSFERS exists.
Table OUTPUTEVENTS exists.
Table CHARTEVENTS exists.


In [21]:
# Load tables into data dict (run AFTER table check)
data = {}
with get_db_connection(db_path) as conn:  # Reuse your context manager
    for tbl in desired_tables:
        if tbl in db_tables:
            df_name = tbl.upper()  
            data[df_name] = pd.read_sql_query(f"SELECT * FROM {tbl}", conn)
            print(f"Loaded {df_name} ({len(data[df_name])} rows, {data[df_name].shape[1]} cols)")

# Access like your CSV vars:
patients = data['PATIENTS']
caregivers = data[ 'CAREGIVERS']
d_items = data['D_ITEMS']
admissions = data['ADMISSIONS']
icustays = data['ICUSTAYS']
noteevents = data['NOTEEVENTS']
microbiologyevents = data['MICROBIOLOGYEVENTS']
transfers = data['TRANSFERS']
outputevents = data['OUTPUTEVENTS']
chartevents = data['CHARTEVENTS']


Loaded PATIENTS (100 rows, 7 cols)
Loaded CAREGIVERS (7567 rows, 3 cols)
Loaded D_ITEMS (12487 rows, 9 cols)
Loaded ADMISSIONS (129 rows, 18 cols)
Loaded ICUSTAYS (136 rows, 11 cols)
Loaded NOTEEVENTS (0 rows, 11 cols)
Loaded MICROBIOLOGYEVENTS (2003 rows, 16 cols)
Loaded TRANSFERS (524 rows, 13 cols)
Loaded OUTPUTEVENTS (11319 rows, 13 cols)
Loaded CHARTEVENTS (758274 rows, 15 cols)


In [22]:
chartevents.head()

,ROW_ID,SUBJECT_ID,HADM_ID,ICUSTAY_ID,ITEMID,CHARTTIME,STORETIME,CGID,VALUE,VALUENUM,VALUEUOM,WARNING,ERROR,RESULTSTATUS,STOPPED
0,5279021,40124,126179,279554,223761,2130-02-04 04:00:00,2130-02-04 04:35:00,19085,95.9,95.9,?F,0.0,0.0,NaN,NaN
1,5279022,40124,126179,279554,224695,2130-02-04 04:25:00,2130-02-04 05:55:00,18999,2222221.7,2222221.7,cmH2O,0.0,0.0,NaN,NaN
2,5279023,40124,126179,279554,220210,2130-02-04 04:30:00,2130-02-04 04:43:00,21452,15.0,15.0,insp/min,0.0,0.0,NaN,NaN
3,5279024,40124,126179,279554,220045,2130-02-04 04:32:00,2130-02-04 04:43:00,21452,94.0,94.0,bpm,0.0,0.0,NaN,NaN
4,5279025,40124,126179,279554,220179,2130-02-04 04:32:00,2130-02-04 04:43:00,21452,163.0,163.0,mmHg,0.0,0.0,NaN,NaN


In [23]:
from datetime import timedelta
from collections import defaultdict

microbiologyevents.columns = microbiologyevents.columns.str.upper()
icustays.columns           = icustays.columns.str.upper()

In [24]:
import pandas as pd

WARD_ID_TARGET = 52
WARD_SIZE = 10  # choose 5/10/20 bucket

microbiologyevents["CHARTDATE"] = pd.to_datetime(microbiologyevents["CHARTDATE"], errors="coerce")

ward52_hadm = icustays.loc[
    (icustays["FIRST_WARDID"] == WARD_ID_TARGET) | (icustays["LAST_WARDID"] == WARD_ID_TARGET),
    "HADM_ID"
].dropna().unique()

ward52_pos = microbiologyevents[
    microbiologyevents["HADM_ID"].isin(ward52_hadm)
    & microbiologyevents["ORG_NAME"].notna()
    & microbiologyevents["CHARTDATE"].notna()
].copy()

ward52_pos["ORG_NAME"] = ward52_pos["ORG_NAME"].astype(str).str.upper().str.strip()
ward52_pos = ward52_pos.sort_values(["ORG_NAME", "CHARTDATE"])

In [25]:
ward52_hadm

array([198503, 157609, 177678, 142345, 155297, 186361, 172454, 195911,
       164869, 188574, 198480, 109698, 114867, 167754, 172082, 171878,
       101361, 174863, 130681, 140372, 124073, 129273, 151798, 175237,
       100375, 176016, 197611, 118192, 142539, 176805, 167612])

In [26]:
episodes_all = []

for org_name, pathogen in PATHOGEN_REGISTRY.items():
    df = ward52_pos[ward52_pos["ORG_NAME"] == org_name.upper()].copy()
    if df.empty:
        continue

    # gap in days between consecutive positives
    df = df.sort_values("CHARTDATE")
    gap_days = df["CHARTDATE"].diff().dt.total_seconds().div(86400)

    # New episode if first row OR gap > window_days
    df["EPISODE_ID"] = (gap_days.isna() | (gap_days > pathogen.time_window_days)).cumsum()

    # Aggregate episode stats
    ep = df.groupby("EPISODE_ID").agg(
        start_time=("CHARTDATE", "min"),
        end_time=("CHARTDATE", "max"),
        culture_events=("ROW_ID", "count"),
        unique_patients=("SUBJECT_ID", "nunique"),
    ).reset_index(drop=True)

    ep["org_name"] = pathogen.org_name
    ep["window_days"] = pathogen.time_window_days
    ep["threshold"] = pathogen.get_ward_threshold(WARD_SIZE)
    ep["alert"] = ep["unique_patients"] >= ep["threshold"]

    episodes_all.append(ep)

episodes_df = pd.concat(episodes_all, ignore_index=True) if episodes_all else pd.DataFrame()


In [27]:
if episodes_df.empty:
    print("No pathogen episodes found in ward 52 for the registry organisms.")
else:
    print("Episodes (counters) found:", len(episodes_df))
    print("Alerts fired (episodes where unique_patients >= threshold):", int(episodes_df["alert"].sum()))

    print("\nAlerts per pathogen:")
    print(
        episodes_df[episodes_df["alert"]]
        .groupby("org_name")
        .agg(
            alert_episodes=("alert", "size"),
            first_alert=("start_time", "min"),
            last_alert=("end_time", "max"),
            max_patients=("unique_patients", "max"),
            threshold=("threshold", "first"),
            window_days=("window_days", "first"),
        )
        .sort_values("alert_episodes", ascending=False)
        .to_string()
    )

    print("\nAll ALERT episodes (each row = one counter instance):")
    print(
        episodes_df[episodes_df["alert"]]
        .sort_values(["org_name", "start_time"])
        .to_string(index=False)
    )


Episodes (counters) found: 45
Alerts fired (episodes where unique_patients >= threshold): 26

Alerts per pathogen:
                                                 alert_episodes first_alert last_alert  max_patients  threshold  window_days
org_name                                                                                                                    
PSEUDOMONAS AERUGINOSA                                        7  2145-12-02 2201-11-16             1          1            1
POSITIVE FOR METHICILLIN RESISTANT STAPH AUREUS               5  2144-07-14 2199-02-11             1          1            1
ESCHERICHIA COLI                                              4  2119-10-22 2169-05-07             1          1            3
STAPH AUREUS COAG +                                           4  2144-07-23 2145-12-02             1          1            1
ACINETOBACTER BAUMANNII COMPLEX                               2  2145-12-06 2145-12-10             1          1            3
KLEBSIELLA

In [34]:
with get_db_connection(db_path) as conn:
    cursor = conn.cursor()
    
    # Create table if it doesn't exist
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS MICROALERTS (
            ALERT_ID INTEGER PRIMARY KEY,
            WARD_ID INTEGER NOT NULL,
            ORG_ID INTEGER,
            ORG_NAME TEXT NOT NULL,
            NUM_PATIENTS INTEGER NOT NULL,
            CULTURE_EVENTS INTEGER NOT NULL,
            START_TIME TIMESTAMP NOT NULL,
            END_TIME TIMESTAMP NOT NULL,
            SEVERITY INTEGER NOT NULL,
            THRESHOLD INTEGER NOT NULL,
            WINDOW_DAYS INTEGER NOT NULL
        )
    ''')
    conn.commit()
    
    # Clear existing records for this ward
    cursor.execute('DELETE FROM MICROALERTS WHERE WARD_ID = ?', (WARD_ID_TARGET,))
    conn.commit()
    
    # Insert alert records from episodes_df
    counter_id = 0
    for idx, row in episodes_df[episodes_df["alert"]].iterrows():
        pathogen = PATHOGEN_REGISTRY[row["org_name"]]
        
        # Convert timestamps to strings
        start_time_str = str(row["start_time"])
        end_time_str = str(row["end_time"])

        cursor.execute(
            '''
            INSERT INTO MICROALERTS 
            (ALERT_ID, WARD_ID, ORG_ID, ORG_NAME, NUM_PATIENTS, CULTURE_EVENTS, 
             START_TIME, END_TIME, SEVERITY, THRESHOLD, WINDOW_DAYS)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ''',
            (
                int(counter_id),
                int(WARD_ID_TARGET),
                int(pathogen.org_id) if pathogen.org_id is not None else None,
                str(row["org_name"]),
                int(row["unique_patients"]),
                int(row["culture_events"]),
                start_time_str,
                end_time_str,
                int(pathogen.danger_weight * 10),
                int(row["threshold"]),
                int(row["window_days"]),
            )
        )
        counter_id += 1
    
    conn.commit()

    # Verify count
    alert_count = cursor.execute(
        'SELECT COUNT(*) FROM MICROALERTS WHERE WARD_ID = ?',
        (WARD_ID_TARGET,)
    ).fetchone()[0]

    # Verify data
    alerts_verify = pd.read_sql_query(
        'SELECT * FROM MICROALERTS WHERE WARD_ID = ? ORDER BY START_TIME',
        conn,
        params=(WARD_ID_TARGET,)
    )

print(f"Created/updated MICROALERTS table with {alert_count} alert records for Ward {WARD_ID_TARGET}")
print(f"\nTotal alerts in database: {len(alerts_verify)}")
print(alerts_verify.head(10))

Created/updated MICROALERTS table with 26 alert records for Ward 52

Total alerts in database: 26
   ALERT_ID  WARD_ID ORG_ID                                         ORG_NAME  \
0         0       52   None                            CLOSTRIDIUM DIFFICILE   
1         3       52   None   ASPERGILLUS SP. NOT FUMIGATUS, FLAVUS OR NIGER   
2         4       52   None                                 ESCHERICHIA COLI   
3         5       52   None                                 ESCHERICHIA COLI   
4        10       52   None  POSITIVE FOR METHICILLIN RESISTANT STAPH AUREUS   
5        22       52   None                              STAPH AUREUS COAG +   
6        23       52   None                              STAPH AUREUS COAG +   
7        24       52   None                              STAPH AUREUS COAG +   
8        15       52   None                           PSEUDOMONAS AERUGINOSA   
9        25       52   None                              STAPH AUREUS COAG +   

   NUM_PATIENTS  CULT

In [35]:
# build a lookup of organism name → org_itemid (int)
microbiologyevents["ORG_NAME"] = (
    microbiologyevents["ORG_NAME"].astype(str)
    .str.upper()
    .str.strip()
)

org_id_map = (
    microbiologyevents
    .dropna(subset=["ORG_NAME", "ORG_ITEMID"])
    .groupby("ORG_NAME")["ORG_ITEMID"]
    .first()          # grab first occurrence for each name
    .astype(int)      # convert float → int
    .to_dict()
)

print(f"{len(org_id_map)} organism names have an ORG_ITEMID; sample:\n",
      list(org_id_map.items())[:8])

# write the ids back into MICROALERTS
with get_db_connection(db_path) as conn:
    cur = conn.cursor()
    for org_name, oid in org_id_map.items():
        cur.execute(
            "UPDATE MICROALERTS SET ORG_ID = ? WHERE ORG_NAME = ?",
            (oid, org_name)
        )
    conn.commit()

    # refresh the dataframe we use for checking
    alerts_verify = pd.read_sql_query(
        'SELECT * FROM MICROALERTS WHERE WARD_ID = ? ORDER BY START_TIME',
        conn,
        params=(WARD_ID_TARGET,)
    )

# optional: keep a local copy named alerts_table too
alerts_table = alerts_verify.copy()

print("after update, alerts table head")
alerts_verify.head()

46 organism names have an ORG_ITEMID; sample:
 [('ACINETOBACTER BAUMANNII', 80194), ('ACINETOBACTER BAUMANNII COMPLEX', 80304), ('ALPHA STREPTOCOCCI', 80052), ('ANAEROBIC GRAM POSITIVE ROD(S)', 80079), ('ASPERGILLUS SP. NOT FUMIGATUS, FLAVUS OR NIGER', 80239), ('BACTEROIDES FRAGILIS GROUP', 80112), ('BETA STREPTOCOCCUS GROUP B', 80045), ('CANDIDA ALBICANS, PRESUMPTIVE IDENTIFICATION', 80254)]
after update, alerts table head


,ALERT_ID,WARD_ID,ORG_ID,ORG_NAME,NUM_PATIENTS,CULTURE_EVENTS,START_TIME,END_TIME,SEVERITY,THRESHOLD,WINDOW_DAYS
0,0,52,80139,CLOSTRIDIUM DIFFICILE,1,1,2105-05-30 00:00:00,2105-05-30 00:00:00,30,1,3
1,3,52,80239,"ASPERGILLUS SP. NOT FUMIGATUS, FLAVUS OR NIGER",1,1,2107-03-23 00:00:00,2107-03-23 00:00:00,15,1,2
2,4,52,80002,ESCHERICHIA COLI,1,14,2119-10-22 00:00:00,2119-10-22 00:00:00,30,1,3
3,5,52,80002,ESCHERICHIA COLI,1,16,2120-08-25 00:00:00,2120-08-25 00:00:00,30,1,3
4,10,52,80293,POSITIVE FOR METHICILLIN RESISTANT STAPH AUREUS,1,1,2144-07-14 00:00:00,2144-07-14 00:00:00,30,1,1
